# Cause Analysis Layer for Port Congestion Risk

This notebook implements the cause analysis layer of the congestion risk framework.  
While the machine learning model predicts whether congestion risk is present, this layer interprets the operational signals responsible for that risk.

The goal is to identify the primary and secondary operational causes of congestion using feature signals derived during the feature engineering stage.

This explanation helps logistics planners understand why congestion risk is occurring before moving to the recommendation stage.

## Importing Libraries

In [1]:
import pandas as pd
import numpy as np

## Dataset Input

The dataset used here is the feature engineered dataset generated in earlier stages.  
It contains operational indicators such as evacuation imbalance, throughput pressure, and gate congestion signals.

These features were also used by the logistic regression model to predict congestion risk.

In [2]:
df_f = pd.read_csv("../../data/JNPA_feature_engineered.csv")

df_f.head()

,Month-Year,Imp_Dwell_Overall,Imp_Dwell_Truck,Imp_Dwell_Train,Parking_Dwell,Imp_Transit_CFS,Imp_Transit_ICD,Exp_Dwell_Overall,Exp_Dwell_Truck,Exp_Dwell_Train,Exp_Transit_CFS,Exp_Transit_ICD,Total_TEUs_Handled,Date,Volume_Lag1,Rail_Friction,Is_Monsoon,Stress,Risk
0,Feb 23,27.8,24.0,69.6,2.55,2.78,79.0,69.5,67.8,80.7,5.06,109.2,467614,2023-02-01,522592.0,2.900000,0,0,1
1,Mar 23,26.5,22.3,63.7,3.65,2.47,97.1,74.0,72.4,86.1,6.42,113.0,560076,2023-03-01,467614.0,2.856502,0,0,0
2,Apr 23,29.9,25.4,53.2,4.00,2.62,90.1,80.0,77.1,104.6,6.46,89.3,503668,2023-04-01,560076.0,2.094488,0,0,1
3,May 23,19.9,17.2,51.0,2.33,2.56,91.1,65.0,62.8,81.3,4.21,109.6,538247,2023-05-01,503668.0,2.965116,0,0,0
4,Jun 23,15.8,13.8,38.4,2.18,2.35,107.0,71.9,69.8,88.5,4.18,112.2,523948,2023-06-01,538247.0,2.782609,1,523948,0


## Operational Signals

The following signals are used for cause analysis:

- Rail_Friction - Rail evacuation imbalance
- Parking_Dwell - Export gate congestion
- Stress - Operational pressure from throughput and seasonal effects
- Volume_Lag1 - Background container demand

In [3]:
signals = df_f[["Rail_Friction","Parking_Dwell","Stress","Volume_Lag1","Risk"]]

signals.head()

,Rail_Friction,Parking_Dwell,Stress,Volume_Lag1,Risk
0,2.900000,2.55,0,522592.0,1
1,2.856502,3.65,0,467614.0,0
2,2.094488,4.00,0,560076.0,1
3,2.965116,2.33,0,503668.0,0
4,2.782609,2.18,523948,538247.0,0


## Risk Categories

Instead of using raw probabilities, congestion risk is interpreted using three categories:

Low Risk → Probability below 0.4  
Moderate Risk → Probability between 0.4 and 0.7  
High Risk → Probability above 0.7  

These categories simplify interpretation for logistics planners.

In [4]:
def risk_category(prob):

    if prob < 0.4:
        return "Low"
    elif prob < 0.7:
        return "Moderate"
    else:
        return "High"

## Signal Normalization

Operational signals exist on different scales.  
To compare their relative strength, signals are normalized using min-max scaling.

This allows the system to rank which operational mechanism is contributing most strongly to congestion risk.

In [5]:
normalized = signals.copy()

for col in ["Rail_Friction","Parking_Dwell","Stress","Volume_Lag1"]:
    
    normalized[col] = (
        signals[col] - signals[col].min()
    ) / (
        signals[col].max() - signals[col].min()
    )

normalized.head()

,Rail_Friction,Parking_Dwell,Stress,Volume_Lag1,Risk
0,0.303212,0.577259,0.000000,0.221291,1
1,0.292779,0.897959,0.000000,0.000000,0
2,0.110006,1.000000,0.000000,0.372167,1
3,0.318831,0.513120,0.000000,0.145120,0
4,0.275055,0.469388,0.731714,0.284304,0


## Operational Hierarchy

When multiple congestion signals appear simultaneously, priority is determined based on operational impact.

Hierarchy of congestion mechanisms:

1. Rail evacuation imbalance (Rail_Friction)
2. Gate congestion (Parking_Dwell)
3. Operational pressure (Stress)
4. Background throughput demand (Volume_Lag1)

Evacuation failures are given the highest priority because they directly block container movement from the port.

In [6]:
hierarchy = ["Rail_Friction","Parking_Dwell","Stress","Volume_Lag1"]

## Cause Identification Logic

The system identifies the strongest operational signal as the primary cause.

If a second signal is at least 70% as strong as the primary signal, it is labeled as a secondary cause.

This avoids producing artificial explanations when only one signal dominates.

In [7]:
def identify_causes(row):

    values = {"Rail_Friction": row["Rail_Friction"],"Parking_Dwell": row["Parking_Dwell"],"Stress": row["Stress"],"Volume_Lag1": row["Volume_Lag1"]}

    ranked = sorted(values.items(), key=lambda x: x[1], reverse=True)

    primary = ranked[0]
    secondary = ranked[1]

    primary_cause = primary[0]

    if secondary[1] >= 0.7 * primary[1]:
        secondary_cause = secondary[0]
    else:
        secondary_cause = None

    return primary_cause, secondary_cause

In [8]:
primary_list = []
secondary_list = []

for _, row in normalized.iterrows():

    primary, secondary = identify_causes(row)

    primary_list.append(primary)
    secondary_list.append(secondary)

df_ca = df_f.copy()

df_ca["Primary_Cause"] = primary_list
df_ca["Secondary_Cause"] = secondary_list

## Operational Interpretation

Each signal corresponds to a real operational mechanism in the port system.

In [9]:
cause_map = {

    "Rail_Friction": "Rail evacuation imbalance",

    "Parking_Dwell": "Export gate congestion",

    "Stress": "Operational throughput pressure",

    "Volume_Lag1": "High container demand"
}

df_ca["Primary_Explanation"] = df_ca["Primary_Cause"].map(cause_map)

df_ca["Secondary_Explanation"] = df_ca["Secondary_Cause"].map(cause_map)



In [10]:

print("Columns being saved:")
print(df_ca.columns)

df_ca.to_csv("../../data/JNPA_cause_analysis_output.csv", index=False)


Columns being saved:
Index(['Month-Year', 'Imp_Dwell_Overall', 'Imp_Dwell_Truck', 'Imp_Dwell_Train',
       'Parking_Dwell', 'Imp_Transit_CFS', 'Imp_Transit_ICD',
       'Exp_Dwell_Overall', 'Exp_Dwell_Truck', 'Exp_Dwell_Train',
       'Exp_Transit_CFS', 'Exp_Transit_ICD', 'Total_TEUs_Handled', 'Date',
       'Volume_Lag1', 'Rail_Friction', 'Is_Monsoon', 'Stress', 'Risk',
       'Primary_Cause', 'Secondary_Cause', 'Primary_Explanation',
       'Secondary_Explanation'],
      dtype='object')


## Example Interpretation

Example congestion explanation produced by this layer:

Congestion Risk: HIGH

Cause Analysis  
Primary Cause: Rail evacuation imbalance  
Secondary Cause: High container throughput

These explanations will be used by the recommendation layer to determine appropriate logistics actions.